In [16]:
import pandas as pd
import tensorflow as tf
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
#from tensorflow.keras.models import load_model
from tensorflow.keras.models import load_model
from art.estimators.classification import KerasClassifier, SklearnClassifier
from art.estimators.classification import XGBoostClassifier
import tensorflow as tf
tf.compat.v1.disable_eager_execution()
import glob
import os

# Path to the folder containing your CSV files
csv_folder = '../code_updated/preprocessed'  # Update this
# Read all CSV files in the folder
csv_files = glob.glob(os.path.join(csv_folder, "*.csv"))
#csv_files = [f for f in csv_files if not os.path.basename(f).rstrip(".csv").endswith("m")]
print(csv_files)



['../code_updated/preprocessed\\csa1.csv', '../code_updated/preprocessed\\csa2.csv', '../code_updated/preprocessed\\csa3.csv', '../code_updated/preprocessed\\fa1.csv', '../code_updated/preprocessed\\fa2.csv', '../code_updated/preprocessed\\fa3.csv', '../code_updated/preprocessed\\mecta.csv', '../code_updated/preprocessed\\msa1.csv', '../code_updated/preprocessed\\msa2.csv', '../code_updated/preprocessed\\msa3.csv', '../code_updated/preprocessed\\rloffa1.csv', '../code_updated/preprocessed\\rloffa2.csv', '../code_updated/preprocessed\\rloffa3.csv', '../code_updated/preprocessed\\rlona1.csv', '../code_updated/preprocessed\\rlona2.csv', '../code_updated/preprocessed\\rlona3.csv']


In [17]:
# Subsets of files
csa = csv_files[:3]
fa = csv_files[3:6]
mecta = csv_files[6:7]
msa = csv_files[7:10]
rloffa = csv_files[10:13]
rlona = csv_files[13:16]
print(csa)
print(fa)
print(mecta)
print(msa)
print(rloffa)
print(rlona)
# Function to load subset and filter normal samples
def load_dataset(files):
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    #df=df[df['Flag'] == 0].copy()
    X = df.drop(columns=['Flag'])   # features only
    y = df['Flag'].values           
    X.columns = [c.replace("[", "_").replace("]", "").replace("<", "_") for c in X.columns]
    return X, y

# Load datasets with labels
X_csa, y_csa = load_dataset(csa)
X_fa, y_fa = load_dataset(fa)
X_mecta, y_mecta = load_dataset(mecta)
X_msa, y_msa = load_dataset(msa)
X_rloffa, y_rloffa = load_dataset(rloffa)
X_rlona, y_rlona = load_dataset(rlona)


# Load models
dnn_model = tf.keras.models.load_model("../models/FP/dnn_model.h5")
dt_model = joblib.load("../models/FP/dt_model.pkl")
rf_model = joblib.load("../models/FP/rf_model.pkl")
et_model = joblib.load("../models/FP/et_model.pkl")
xgb_model = joblib.load("../models/FP/xgboost_model.pkl")

# Wrap models
dnn_art = KerasClassifier(model=dnn_model, clip_values=(0, 255))
#dt_art = SklearnClassifier(model=dt_model, clip_values=(0, 255))
#rf_art = SklearnClassifier(model=rf_model, clip_values=(0, 255))
#et_art = SklearnClassifier(model=et_model, clip_values=(0, 255))
#xgb_art = XGBoostClassifier(model=xgb_model, clip_values=(0, 255))


['../code_updated/preprocessed\\csa1.csv', '../code_updated/preprocessed\\csa2.csv', '../code_updated/preprocessed\\csa3.csv']
['../code_updated/preprocessed\\fa1.csv', '../code_updated/preprocessed\\fa2.csv', '../code_updated/preprocessed\\fa3.csv']
['../code_updated/preprocessed\\mecta.csv']
['../code_updated/preprocessed\\msa1.csv', '../code_updated/preprocessed\\msa2.csv', '../code_updated/preprocessed\\msa3.csv']
['../code_updated/preprocessed\\rloffa1.csv', '../code_updated/preprocessed\\rloffa2.csv', '../code_updated/preprocessed\\rloffa3.csv']
['../code_updated/preprocessed\\rlona1.csv', '../code_updated/preprocessed\\rlona2.csv', '../code_updated/preprocessed\\rlona3.csv']


In [18]:
print(X_rlona.head(2))

   CAN ID  DLC  DATA_0  DATA_1  DATA_2  DATA_3  DATA_4  DATA_5  DATA_6  DATA_7
0     737    8       0       0       0       0       0       0       0       4
1     852    8      31     255      64       0       0       4       9       0


In [19]:
from art.attacks.evasion import FastGradientMethod, BasicIterativeMethod, ProjectedGradientDescent

def generate_constrained_attack(estimator, X, method="FGSM", eps=1.0, eps_step=0.1, max_iter=10):
    """
    Generate adversarial examples with IVN constraints using FGSM, BIM, or PGD.
    
    Parameters:
    - estimator: ART classifier
    - X: np.ndarray or pd.DataFrame, shape (n_samples, 10)
    - method: str, one of {"FGSM", "BIM", "PGD"}
    - eps: float, total allowed perturbation
    - eps_step: float, step size
    - max_iter: int, only used for iterative attacks (BIM, PGD)
    
    Returns:
    - X_adv: adversarial examples (np.ndarray), clipped to [0, 255] on DATA[0]-[7]
    """
    if isinstance(X, pd.DataFrame):
        X = X.to_numpy()

    # Only allow perturbation on DATA[0]-[7]
    perturbation_mask = np.array([False, False] + [True] * 8)

    # Select the attack method
    if method == "FGSM":
        attack = FastGradientMethod(estimator=estimator, eps=eps, eps_step=eps_step)
    elif method == "BIM":
        attack = BasicIterativeMethod(estimator=estimator, eps=eps, eps_step=eps_step, max_iter=max_iter, verbose=False)
    elif method == "PGD":
        attack = ProjectedGradientDescent(estimator=estimator, eps=eps, eps_step=eps_step, max_iter=max_iter, num_random_init=1, verbose=False)
    else:
        raise ValueError("Unsupported attack method. Choose 'FGSM', 'BIM', or 'PGD'.")

    # Generate adversarial examples
    X_adv = attack.generate(x=X, mask=perturbation_mask)

    # Clip DATA[0] to DATA[7] to valid byte range
    X_adv[:, 2:] = np.clip(X_adv[:, 2:], 0, 255)
    
    # Log attack details
    print(f"[{method}] Generated adversarial examples with eps={eps}, eps_step={eps_step}, max_iter={max_iter if method != 'FGSM' else 'N/A'}")
    
    return X_adv


In [20]:
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.metrics import matthews_corrcoef

def constraint_compliant(X):
    X_np = X.to_numpy() if isinstance(X, pd.DataFrame) else X
    return (
        (X_np[:, 0] >= 0) & (X_np[:, 0] <= 1068) &  # CAN ID
        (X_np[:, 1] >= 1) & (X_np[:, 1] <= 8) &     # DLC
        np.all((X_np[:, 2:] >= 0) & (X_np[:, 2:] <= 255), axis=1)  # DATA[0]-[7]
    )

def evaluate_model_on_adversarial_tabular(
    model,
    X_clean,
    y_clean,          # <-- now you pass true labels (0=normal, 1=attack)
    X_adv_dict,
    model_name="RF",
    attack_name="BIM",
    is_probabilistic=False
):
    results = []
    sample_size = len(X_clean)

    # Predict on clean data
    if is_probabilistic:
        y_prob = model.predict(X_clean).ravel()   # values in (0, 1)
        y_pred_clean = (y_prob >= 0.5).astype(int) #y_pred_clean = np.argmax(model.predict(X_clean), axis=1)
    else:
        y_pred_clean = model.predict(X_clean)

    # Confusion matrix (TN, FP, FN, TP)
    cm_clean = confusion_matrix(y_clean, y_pred_clean, labels=[0,1])
    tn, fp, fn, tp = cm_clean.ravel()
    f1_clean = f1_score(y_clean, y_pred_clean, average='weighted')
    mcc_clean = matthews_corrcoef(y_clean, y_pred_clean)

    for eps_val, X_adv in X_adv_dict.items():
        valid_mask = constraint_compliant(X_adv)
        X_adv_valid = X_adv[valid_mask]
        if len(X_adv_valid) == 0:
            continue

        # Ground truth is same as y_clean for aligned samples
        y_true_adv = y_clean[valid_mask]

        if is_probabilistic:
            y_prob_adv = model.predict(X_adv_valid).ravel()   # values in (0, 1)
            y_pred_adv = (y_prob_adv >= 0.5).astype(int)
            #y_pred_adv = np.argmax(model.predict(X_adv_valid), axis=1)
            y_pred_adv_binary = (y_pred_adv.ravel() >= 0.5).astype(int)
        else:
            y_pred_adv = model.predict(X_adv_valid)
            y_pred_adv_binary = y_pred_adv.astype(int)


        #y_pred_adv_binary = (y_pred_adv.ravel() >= 0.5).astype(int)
        #y_pred_adv_binary = (y_pred_adv != 0).astype(int)
        f1_adv = f1_score(y_true_adv, y_pred_adv_binary, average='weighted')

                # Confusion matrix for adversarial data
        cm_adv = confusion_matrix(y_true_adv, y_pred_adv_binary, labels=[0,1])
        tn_adv, fp_adv, fn_adv, tp_adv = cm_adv.ravel()
        mcc_adv = matthews_corrcoef(y_true_adv, y_pred_adv_binary)
        asr  = (fp_adv + fn_adv) / (tp_adv + tn_adv + fp_adv + fn_adv)


        save_dir = "../results/saved_flags"
        os.makedirs(save_dir, exist_ok=True)

        # filenames include model, attack, epsilon
        np.save(os.path.join(save_dir, f"{model_name}_{attack_name}_{eps_val}_y_clean.npy"), y_clean)
        np.save(os.path.join(save_dir, f"{model_name}_{attack_name}_{eps_val}_y_pred_clean.npy"), y_pred_clean)
        np.save(os.path.join(save_dir, f"{model_name}_{attack_name}_{eps_val}_y_true_adv.npy"), y_true_adv)
        np.save(os.path.join(save_dir, f"{model_name}_{attack_name}_{eps_val}_y_pred_adv_binary.npy"), y_pred_adv_binary)

        #mcc_clean=0
        #mcc_adv=0
        #fn_adv=0



        results.append([
            model_name,                # Model
            sample_size,               # Sample Size
            f"{f1_clean*100:.1f}%",    # F1 Score
            f"{mcc_clean:.3f}",        # MCC
            fp,
            fn,                        # FN (clean)
            attack_name,               # Attack
            eps_val,                   # Epsilon
            f"{f1_adv*100:.1f}%",      # F1 Score (Adv)
            f"{mcc_adv:.3f}",          # MCC(Adv)
            fp_adv,
            fn_adv,                    # FN (Adv)
            f"{asr:.1%}"               # ASR (FN)
        ])

    columns = [
        "Model", "Sample Size",
        "F1 Score", "MCC", "FP", "FN",
        "Attack", "Epsilon",
        "F1 Score (Adv)", "MCC (Adv)", "FP (Adv)","FN (Adv)", "ASR"
    ]

    return pd.DataFrame(results, columns=columns)


In [22]:
def run_attacks(dataset_name, X_normal, y_clean, models, attack_methods):
    results = {}
    cols = X_normal.columns
    for attack in attack_methods:
        X_adv_dict_all = {}  # holds adversarial sets per model

        # === Generate adversarial examples ===
        if attack in ["FGSM", "BIM", "PGD"]:
            # These only use the DNN
            if attack == "FGSM":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="FGSM", eps=1.0, eps_step=0.1),
                    5: generate_constrained_attack(dnn_art, X_normal, method="FGSM", eps=5.0, eps_step=0.1)
                }
            elif attack == "BIM":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="BIM", eps=1.0, eps_step=0.1, max_iter=10),
                    5: generate_constrained_attack(dnn_art, X_normal, method="BIM", eps=5.0, eps_step=0.1, max_iter=10)
                }
            elif attack == "PGD":
                X_adv_dict = {
                    1: generate_constrained_attack(dnn_art, X_normal, method="PGD", eps=1.0, eps_step=0.1, max_iter=20),
                    5: generate_constrained_attack(dnn_art, X_normal, method="PGD", eps=5.0, eps_step=0.1, max_iter=20)
                }

            # store once, reused by all models
            X_adv_dict_all = {"DNN": X_adv_dict}

        elif attack == "DT":
            # Generate DT attack for *every* model in models
            for model_name, model_obj in models.items():
                if model_name == "DNN":
                    base_model = dnn_model
                else:
                    base_model = model_obj

                X_adv_dict_all[model_name] = {
                    1: generate_dt_attack(base_model, X_normal, target_class=1, epsilon=1.0, batch_size=1024),
                    5: generate_dt_attack(base_model, X_normal, target_class=1, epsilon=5.0, batch_size=1024)
                }

        else:
            continue
    
        # === Evaluate across all models ===
        for model_name, model_obj in models.items():
            X_fixed = pd.DataFrame(X_normal.values, columns=cols)

            # choose the right adversarial examples
            if attack == "DT":
                X_adv_fixed = {eps: pd.DataFrame(X_adv, columns=cols)
                               for eps, X_adv in X_adv_dict_all[model_name].items()}
            else:
                X_adv_fixed = {eps: pd.DataFrame(X_adv, columns=cols)
                               for eps, X_adv in X_adv_dict_all["DNN"].items()}

            result_key = f"{dataset_name}_{model_name}_{attack}"
            results[result_key] = evaluate_model_on_adversarial_tabular(
                model_obj,
                X_fixed,
                y_clean,
                X_adv_fixed,
                model_name=model_name,
                attack_name=attack,
                is_probabilistic=(model_name == "DNN")
            )

    return results


In [23]:
models = {"DNN": dnn_art, "DT": dt_model, "RF": rf_model, "ET": et_model, "XGBoost": xgb_model}

datasets = {"rlona": (X_rlona, y_rlona), "rloffa": (X_rloffa, y_rloffa), "msa": (X_msa, y_msa), "mecta": (X_mecta, y_mecta), "fa": (X_fa, y_fa), "csa": (X_csa,y_csa)}

attack_methods = ["FGSM", "BIM", "PGD"]


In [24]:
for dataset_key, (X,y) in datasets.items():
    print(y)

[0 0 0 ... 0 0 0]
[1 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]
[0 0 0 ... 0 0 0]


In [25]:
# mapping dataset keys to pretty names
dataset_labels = {
    "csa": "Correlated Signal Attack",
    "fa": "Fuzzing Attack",
    "mecta": "Max Engine Coolant Temp Attack",
    "msa": "max_speedometer_attack",  
    "rloffa": "Reverse Light Off Attack",
    "rlona": "Reverse Light On Attack"
}

# Run attacks for all datasets
all_results_fp = {}
for dataset_key, (X,y) in datasets.items():
    dataset_results = run_attacks(dataset_key, X, y, models, attack_methods)

    # merge all model/attack results into one big DataFrame for this dataset
    merged_df = pd.concat(dataset_results.values(), ignore_index=True)

    # store under dataset key
    all_results_fp[dataset_key] = merged_df



c:\Users\Lenovo\anaconda3\envs\ivn-ids\lib\site-packages\keras\engine\training_v1.py:2357: UserWarning: `Model.state_updates` will be removed in a future version. This property should not be used in TensorFlow 2.0, as `updates` are applied automatically.
  updates=self.state_updates,


[FGSM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=N/A
[FGSM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=N/A
[BIM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=10
[BIM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=10
[PGD] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=20
[PGD] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=20
[FGSM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=N/A
[FGSM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=N/A
[BIM] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=10
[BIM] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=10
[PGD] Generated adversarial examples with eps=1.0, eps_step=0.1, max_iter=20
[PGD] Generated adversarial examples with eps=5.0, eps_step=0.1, max_iter=20
[FGSM] Generated adversarial examples with eps=1.0, eps_step=0.1, ma

In [26]:
for dataset_key, df in all_results_fp.items():
   dataset_label = dataset_labels.get(dataset_key, dataset_key)
   print(f"\n=== {dataset_label} ===")
   display(df)   # Jupyter; if terminal: print(df.to_string())


=== Reverse Light On Attack ===


,Model,Sample Size,F1 Score,MCC,FP,FN,Attack,Epsilon,F1 Score (Adv),MCC (Adv),FP (Adv),FN (Adv),ASR
0,DNN,495216,98.8%,0.797,1692,3924,FGSM,1,98.5%,0.819,423,4434,1.4%
1,DNN,495216,98.8%,0.797,1692,3924,FGSM,5,93.3%,0.120,7611,13389,6.0%
2,DT,495216,99.6%,0.930,1893,306,FGSM,1,93.7%,0.117,2667,14175,4.9%
3,DT,495216,99.6%,0.930,1893,306,FGSM,5,93.6%,0.099,3762,14157,5.1%
4,RF,495216,99.6%,0.930,1896,300,FGSM,1,94.1%,0.206,846,13974,4.3%
5,RF,495216,99.6%,0.930,1896,300,FGSM,5,94.0%,0.161,1125,14205,4.4%
6,ET,495216,99.6%,0.930,1893,306,FGSM,1,98.8%,0.851,741,3333,1.2%
7,ET,495216,99.6%,0.930,1893,306,FGSM,5,94.2%,0.263,3,14088,4.0%
8,XGBoost,495216,99.6%,0.930,1827,354,FGSM,1,93.7%,0.112,2661,14214,4.9%
9,XGBoost,495216,99.6%,0.930,1827,354,FGSM,5,93.5%,0.093,3702,14226,5.1%



=== Reverse Light Off Attack ===


,Model,Sample Size,F1 Score,MCC,FP,FN,Attack,Epsilon,F1 Score (Adv),MCC (Adv),FP (Adv),FN (Adv),ASR
0,DNN,190962,95.9%,0.612,0,6495,FGSM,1,94.4%,0.371,0,5472,4.1%
1,DNN,190962,95.9%,0.612,0,6495,FGSM,5,93.4%,0.153,1047,5838,5.1%
2,DT,190962,97.2%,0.733,18,4734,FGSM,1,94.5%,0.340,627,5169,4.4%
3,DT,190962,97.2%,0.733,18,4734,FGSM,5,94.8%,0.401,156,5169,4.0%
4,RF,190962,97.2%,0.738,18,4656,FGSM,1,94.9%,0.443,0,5085,3.8%
5,RF,190962,97.2%,0.738,18,4656,FGSM,5,94.9%,0.439,0,5106,3.8%
6,ET,190962,97.2%,0.733,18,4734,FGSM,1,94.8%,0.423,33,5163,3.9%
7,ET,190962,97.2%,0.733,18,4734,FGSM,5,94.8%,0.413,84,5169,3.9%
8,XGBoost,190962,97.3%,0.742,24,4593,FGSM,1,95.0%,0.450,3,5040,3.8%
9,XGBoost,190962,97.3%,0.742,24,4593,FGSM,5,95.0%,0.439,66,5040,3.8%



=== max_speedometer_attack ===


...

,Model,Sample Size,F1 Score,MCC,FP,FN,Attack,Epsilon,F1 Score (Adv),MCC (Adv),FP (Adv),FN (Adv),ASR
0,DNN,58020,99.8%,-0.000,1,88,FGSM,1,100.0%,0.000,0,6,0.0%
1,DNN,58020,99.8%,-0.000,1,88,FGSM,5,99.5%,-0.001,389,6,1.0%
2,DT,58020,99.9%,0.748,16,26,FGSM,1,100.0%,-0.000,9,6,0.0%
3,DT,58020,99.9%,0.748,16,26,FGSM,5,99.9%,-0.000,29,6,0.1%
4,RF,58020,99.9%,0.748,16,26,FGSM,1,100.0%,0.000,0,6,0.0%
5,RF,58020,99.9%,0.748,16,26,FGSM,5,100.0%,0.000,0,6,0.0%
6,ET,58020,99.9%,0.733,16,28,FGSM,1,100.0%,-0.000,8,6,0.0%
7,ET,58020,99.9%,0.733,16,28,FGSM,5,100.0%,0.000,0,6,0.0%
8,XGBoost,58020,99.9%,0.733,16,28,FGSM,1,100.0%,-0.000,8,6,0.0%
9,XGBoost,58020,99.9%,0.733,16,28,FGSM,5,100.0%,-0.000,19,6,0.1%



=== Fuzzing Attack ===


,Model,Sample Size,F1 Score,MCC,FP,FN,Attack,Epsilon,F1 Score (Adv),MCC (Adv),FP (Adv),FN (Adv),ASR
0,DNN,88968,99.9%,0.978,47,0,FGSM,1,100.0%,0.992,10,0,0.0%
1,DNN,88968,99.9%,0.978,47,0,FGSM,5,99.5%,0.803,341,0,0.6%
2,DT,88968,100.0%,0.990,22,0,FGSM,1,99.6%,0.838,262,0,0.4%
3,DT,88968,100.0%,0.990,22,0,FGSM,5,99.6%,0.789,117,140,0.4%
4,RF,88968,100.0%,0.989,23,0,FGSM,1,100.0%,0.998,2,0,0.0%
5,RF,88968,100.0%,0.989,23,0,FGSM,5,100.0%,0.994,7,0,0.0%
6,ET,88968,100.0%,0.992,18,0,FGSM,1,100.0%,1.000,0,0,0.0%
7,ET,88968,100.0%,0.992,18,0,FGSM,5,100.0%,1.000,0,0,0.0%
8,XGBoost,88968,100.0%,0.990,22,0,FGSM,1,100.0%,0.994,8,0,0.0%
9,XGBoost,88968,100.0%,0.990,22,0,FGSM,5,99.9%,0.948,70,0,0.1%



=== Correlated Signal Attack ===


,Model,Sample Size,F1 Score,MCC,FP,FN,Attack,Epsilon,F1 Score (Adv),MCC (Adv),FP (Adv),FN (Adv),ASR
0,DNN,180905,99.8%,0.970,337,0,FGSM,1,100.0%,0.000,18,0,0.0%
1,DNN,180905,99.8%,0.970,337,0,FGSM,5,99.9%,0.000,206,0,0.2%
2,DT,180905,100.0%,0.992,86,0,FGSM,1,100.0%,0.000,91,0,0.1%
3,DT,180905,100.0%,0.992,86,0,FGSM,5,99.9%,0.000,309,0,0.3%
4,RF,180905,100.0%,0.992,90,0,FGSM,1,100.0%,0.000,34,0,0.0%
5,RF,180905,100.0%,0.992,90,0,FGSM,5,100.0%,0.000,40,0,0.0%
6,ET,180905,100.0%,0.992,82,0,FGSM,1,100.0%,0.000,19,0,0.0%
7,ET,180905,100.0%,0.992,82,0,FGSM,5,99.9%,0.000,207,0,0.2%
8,XGBoost,180905,99.9%,0.988,127,0,FGSM,1,100.0%,0.000,79,0,0.1%
9,XGBoost,180905,99.9%,0.988,127,0,FGSM,5,99.9%,0.000,297,0,0.2%


In [ ]:
for key, df in all_results_fp.items():
    df.to_csv(f"results/{key}.csv", index=False)

In [28]:
# #from sklearn.metrics import matthews_corrcoef

# import os
# import numpy as np
# import pandas as pd
# from sklearn.metrics import matthews_corrcoef

# # where the arrays live
# FLAGS_DIR = "../results/saved_flags"  # change if needed

# def _load_pair(model, attack, eps, name):
#     """Load FP and FN arrays and concatenate them."""
#     fp = os.path.join(FLAGS_DIR, f"{model}_{attack}_{eps}_{name}_fp.npy")
#     fn = os.path.join(FLAGS_DIR, f"{model}_{attack}_{eps}_{name}_fn.npy")
#     a = np.load(fp).ravel()
#     b = np.load(fn).ravel()
#     return np.concatenate([a, b])

# def add_mcc_columns(df):
#     df = df.copy()
#     df["MCC"] = np.nan
#     df["MCC (Adv)"] = np.nan

#     for (m, atk, eps), idx in df.groupby(["Model", "Attack", "Epsilon"]).groups.items():
#         # load merged flags
#         y_clean = _load_pair(m, atk, eps, "y_clean")
#         y_pred_clean = _load_pair(m, atk, eps, "y_pred_clean")
#         y_true_adv = _load_pair(m, atk, eps, "y_true_adv")
#         y_pred_adv_binary = _load_pair(m, atk, eps, "y_pred_adv_binary")

#         # compute MCCs
#         mccb = matthews_corrcoef(y_clean, y_pred_clean)
#         mccm = matthews_corrcoef(y_true_adv, y_pred_adv_binary)

#         # write the same MCCs to all rows for this (Model, Attack, Epsilon)
#         df.loc[idx, "MCC"] = mccb
#         df.loc[idx, "MCC (Adv)"] = mccm

#     return df

# # apply to every dataset table you print
# all_results_with_mcc = {}
# for dataset_key, df in all_results_fp.items():
#     df_mcc = add_mcc_columns(df)
#     col_order = [
#         "FP",
#         "FP (Adv)", "ASR (FP)"
#     ]
#     df_mcc=df_mcc[col_order]
#     all_results_with_mcc[dataset_key] = df_mcc
#     dataset_label = dataset_labels.get(dataset_key, dataset_key)
#     filename = f"{dataset_key}_results_fp.tex"
#     df_mcc.to_latex(filename, index=False, escape=False)
#     print(f"\n=== {dataset_label} ===")
#     display(df_mcc)
